In [12]:
from pathlib import Path

import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [3]:
BASE = Path(r"G:\My Drive\HFT\Capital Stake Day\parsed\2026-06-30")

F_SNAPSHOT = BASE / "2026-06-30_ob_snapshot.parquet"
F_UPDATES  = BASE / "2026-06-30_ob_updates.parquet"
F_TRADES   = BASE / "2026-06-30_trades.parquet"

SYMBOL = "MCB"

In [5]:
def read_symbol_chunked(path: Path, symbol: str, batch_size: int = 500_000) -> pd.DataFrame:
    """Stream the file batch-by-batch, filter each batch, concat survivors.
    Peak memory ≈ one batch + accumulated MCB rows."""
    pf = pq.ParquetFile(path)
    kept = []
    for batch in pf.iter_batches(batch_size=batch_size):
        mask = pa.compute.equal(batch.column("symbol"), symbol)
        filtered = batch.filter(mask)
        if filtered.num_rows:
            kept.append(filtered)
    if not kept:
        # empty frame with correct schema
        return pf.schema_arrow.empty_table().to_pandas()
    return pa.Table.from_batches(kept).to_pandas()

df_trades   = read_symbol(F_TRADES,   SYMBOL)
df_updates  = read_symbol(F_UPDATES,  SYMBOL)
df_snapshot = read_symbol(F_SNAPSHOT, SYMBOL)

print(f"trades:   {df_trades.shape}")
print(f"updates:  {df_updates.shape}")
print(f"snapshot: {df_snapshot.shape}")

trades:   (864, 20)
updates:  (2294, 22)
snapshot: (64721, 27)


In [17]:
df_trades.to_csv(os.path.join(BASE, f"trades_{SYMBOL}.csv"))
df_updates.to_csv(os.path.join(BASE, f"Ob_updates_{SYMBOL}.csv"))
df_snapshot.to_csv(os.path.join(BASE, f"Ob_snapshot_{SYMBOL}.csv"))

